In [1]:
import numpy as np
import pathlib as P
import pickle
import pandas as pd

In [2]:
ns = ["cc", "mf", "bp"]
ontology = ["cellular_component", "molecular_function", "biological_process"]

In [3]:
def align_labels(predicted_results, o1, o2, fill_value=0.0):
    """
    Align predicted results to match new label order, handling missing labels.
    
    Args:
        predicted_results: numpy array of shape (N, len(o1))
        o1: list - original label order
        o2: list - desired label order (can have different labels/length)
        fill_value: value to use for labels in o2 that don't exist in o1
    
    Returns:
        numpy array of shape (N, len(o2)) - aligned predictions
    """
    N = predicted_results.shape[0]
    M_new = len(o2)
    
    # Initialize new result array
    aligned_results = np.full((N, M_new), fill_value, dtype=predicted_results.dtype)
    
    # Create mapping from label to column index in o1
    o1_label_to_idx = {label: idx for idx, label in enumerate(o1)}
    
    # Fill the aligned results
    for new_idx, label in enumerate(o2):
        if label in o1_label_to_idx:
            old_idx = o1_label_to_idx[label]
            aligned_results[:, new_idx] = predicted_results[:, old_idx]
        # If label not in o1, keep the fill_value (already set)
    
    return aligned_results

In [4]:
# previous order of labels
namespace_path = P.Path("/data0/shaojiangyi/pprogo-flg-1/data/namespace_terms.pkl")
with open(namespace_path, "rb") as f:
    namespace_terms = pickle.load(f)
prep_labels = [namespace_terms[i] for i in ontology]

In [5]:
# # current order of labels
# root_path = P.Path("/data0/zhaojianxiang/data/")
# curr_labels = []
# for n in ns:
#     label_path = root_path / n / "go_list.txt"
#     with open(label_path, "r") as f:
#         labels = f.read().splitlines()
#     curr_labels.append(labels)
lable_path = [f"/data0/shaojiangyi/pprogo-flg-1/results/union_space_preds_only1/{x}/test/union_go_terms.npy" for x in ns]
curr_labels = [np.load(p, allow_pickle=True).tolist() for p in lable_path]

In [6]:
[print(len(l)) for l in curr_labels]
[print(len(l)) for l in prep_labels]

2881
6860
21822
2880
6854
21814


[None, None, None]

In [7]:
# # deepfri
# root_path = P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-deepfri/data-netgo")
# pred_paths = [root_path / ns[i] / f"{ns[i]}_result_pred.npy" for i in range(len(ns))]
# preds = [np.load(p) for p in pred_paths]
# # Align predictions to the new label order
# aligned_preds = [np.stack((align_labels(pred[0], prep_labels[i], curr_labels[i]),
#                            align_labels(pred[1], prep_labels[i], curr_labels[i]))) 
#                  for i, pred in enumerate(preds)]
# # print shape
# print([p.shape for p in aligned_preds])
# # Save aligned predictions
# for i, pred in enumerate(aligned_preds):
#     np.save(root_path / f"{ns[i]}_result_aligned.npy", pred)

In [8]:
root_paths = [
    P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-deepgoplus/data-netgo"),
    P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-tale/data-netgo"),
    P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-deepfri/data-netgo"),
    P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-gcn/"),
    P.Path("/data0/shaojiangyi/pprogo-flg-1/netgo_benchmark/netgo-pfresgo/data-netgo"),
    P.Path("/data0/shaojiangyi/dpfunc/netgo-results"),
    P.Path("/data0/shaojiangyi/GOA-S2F/data-netgo/ng-v1-25-03-26"),
]
for root_path in root_paths:
    if root_path.name == "netgo-gcn":
        pred_paths = [root_path / f"{ns[i]}_result_pred.npy" for i in range(len(ns))]
    else:
        pred_paths = [P.Path(root_path) / ns[i] / f"{ns[i]}_result_pred.npy" for i in range(len(ns))]
    preds = [np.load(p) for p in pred_paths]
    # Align predictions to the new label order
    aligned_preds = [np.stack((align_labels(pred[0], prep_labels[i], curr_labels[i]),
                               align_labels(pred[1], prep_labels[i], curr_labels[i]))) 
                     for i, pred in enumerate(preds)]
    # print shape
    print([p.shape for p in aligned_preds])
    # Save aligned predictions
    for i, pred in enumerate(aligned_preds):
        np.save(P.Path(root_path) / f"{ns[i]}_result_aligned.npy", pred)


[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
[(2, 268, 2881), (2, 505, 6860), (2, 491, 21822)]
